# broadcasting-rules — worked example 2: Z-score each feature column with broadcast subtract and divide

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcasting-rules`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When `x` is `(N, D)` and a per-feature statistic is `(D,)`, right-align broadcasting matches the trailing `D` axis automatically — the `(D,)` vector is treated as `(1, D)` and stretched across the `N` rows. This is the row-broadcast pattern, and it works for subtraction and division the same way it works for addition.

## Worked solution

Goal: standardize each **feature column** of a batch `x` of shape `(N, D)` to zero mean and unit variance.

**Step 1 — compute per-feature statistics.** `mu = x.mean(dim=0)` reduces over the `N` (row) axis, leaving one mean per feature: shape `(D,)`. Same for `sigma = x.std(dim=0)`, also `(D,)`. We reduce over `dim=0` because each *column* is one feature.

**Step 2 — subtract the mean via row broadcast.** `x - mu` right-aligns `(N, D)` against `(D,)`. The `(D,)` left-pads to `(1, D)` and stretches across all `N` rows, so feature `d` gets `mu[d]` subtracted from every row. No `unsqueeze` needed — the trailing axes already line up.

**Step 3 — divide by the std the same way.** `(x - mu) / sigma` broadcasts `sigma` of shape `(D,)` identically. Each column is scaled by its own standard deviation.

**Step 4 — why this is safe.** Because the statistic axis (`D`) is the *trailing* axis, broadcasting targets the right axis with zero reshaping. If the statistic had been per-sample (`(N,)`) we would have needed `[:, None]` instead — that is the column-broadcast trap, which does not apply here.

In [ ]:
def standardize_columns(x):
    mu = x.mean(dim=0)       # (D,)
    sigma = x.std(dim=0)     # (D,)
    return (x - mu) / sigma

t.manual_seed(0)
x = t.randn(5, 3) * 4 + 10   # non-trivial mean/scale per column
z = standardize_columns(x)
print('output shape   =', tuple(z.shape))
print('col means ~0   =', t.round(z.mean(dim=0), decimals=4).tolist())
print('col stds  ~1   =', t.round(z.std(dim=0), decimals=4).tolist())